# Spatial acessibility and interaction through networks

In spatial data science, we use [spatial
graphs](../spatial_graphs/hands_on.qmd) to reflect the degree of spatial
interaction between observations. You already know how to represent
interaction based on contiguity, distances or spatial regimes. However,
in many contexts (notably urban), the interaction follows spatially
restricted movement patterns - like those along street networks. In this
session, we will cover it, both as a movement through the streets and
the one depending on public transport.

In [ ]:
import datetime

import geopandas as gpd
import numpy as np
import pandarm
import r5py
import shapely

from libpysal import graph, kernels

We start with the data reflecting Census areas in Prague as we did in
the last session. The dataset contains relatively granular spatial units
with an information about the population living in each, based on 2021
data.

In [ ]:
prague = gpd.read_file(
    "https://martinfleischmann.net/sds/spatial_graphs/data/zsj_prague_2021.gpkg"
)
prague = prague.set_index("NAZ_ZSJ")
prague.explore()

## Walking through the streets

First, we will rely on the `pandarm` package to model interaction
reflecting movement (walking, driving, biking) through the streets.

All parts of the analysis here require you to build a routable network.
While you can do that manually from sets of nodes and edges (with edge
weights), the most common application is to rely on the street network
defined by OpenStreetMap. Given a `GeoDataFrame`, `pandarm` can fetch
the data and construct such a network itself, using
`pandarm.Network.from_gdf`, as shown below. However, this uses Overpass
API, which is currently struggling to keep up the demand (and block
bots), so to avoid additional requests, the code below was used to
download, construct and save the network to HDF5 file, suitable for
multi-component data like this.

``` py
network = pandarm.Network.from_gdf(prague, network_type="walk")
network.save_hdf5("data/prague_walk.h5")
```

We can load the saved file directly with the `pandarm`. Download the
[prague_walk.h5](https://martinfleischmann.net/sds/spatial_networks/data/prague_walk.h5)
file and put it to the folder `data` (or change the path below).

In [ ]:
network = pandarm.Network.from_hdf5("data/prague_walk.h5")

In [ ]:
# temp fix
network.crs = prague.crs
network.edges_df = network.edges_df.set_crs(prague.crs, allow_override=True)
network.nodes_df = network.nodes_df.set_crs(prague.crs, allow_override=True)

The network is, in principle, composed of two GeoDataFrames, one
capturing edges (streets) and another one capturing nodes
(intersections).

Edges:

In [ ]:
network.edges_df.head()

Nodes:

In [ ]:
network.nodes_df.head()

Nodes have a special role in the `pandarm.Network`, as the analysis
outcome is typically linked to node geometry, as we’ll see below.

Second dataset used in this session captures locations of public and
private schools across Prague, originating from the
[GeoPortalPraha.cz](https://geoportalpraha.cz/data-a-sluzby/5d32db18a5d545a79ad3029917984a89).
Read the data and familiarise yourself with it.

In [ ]:
schools = gpd.read_file(
    'https://martinfleischmann.net/sds/spatial_networks/data/schools.zip'
)
schools.head()

To work with the school locatoins within the network, we need to assign
them onto the network using the `set_pois` (set Points of Interest)
method.

In [ ]:
network.set_pois(
    category="schools",
    maxdist=2000,
    maxitems=5,
    x_col=schools.geometry.x,
    y_col=schools.geometry.y,
)

### Nearest POIs

Using this, you can find the distance nearest schools. At most, you can
find the distance to 5 nearest specified above.

In [ ]:
nearest_schools = network.nearest_pois(
    distance=2000,
    category="schools", 
    num_pois=5,
)
nearest_schools.head()

The distance is measured from every single node in the network and can
be linked back to the `nodes_df`. That allows you to easily visualise
the outcome - the map of distance to the nearest school (of any kind)
when walking along the street network.

In [ ]:
ax = network.nodes_df.plot(nearest_schools[1], markersize=.01, legend=True)
ax.set_axis_off()

For a better illustration, lets use a subset of specific schools only.
Filter out `'Základní umělecká škola'` which are art schools.

In [ ]:
art_school = schools.query("TYP_TXT == 'Základní umělecká škola'")
network.set_pois(
    category="art_school", 
    maxdist=5000,
    maxitems=1,
    x_col=art_school.geometry.x, 
    y_col=art_school.geometry.y,
)

Get the distances to nearest art school. This time, fetch also the ID of
the PoI, not just the distance.

In [ ]:
nearest_art_school = network.nearest_pois(
    distance=5000, 
    category="art_school", 
    include_poi_ids=True,
)
nearest_art_school.head()

The resulting DataFrame now contains two columns, first one capturing
the distance, and the second one capturing the ID of the nearest school.
The ID is an integer position, not the index this time.

In [ ]:
ax = art_school.plot(markersize=1, color='orange', zorder=2)
network.nodes_df.plot(
    nearest_art_school[1], markersize=.01, legend=True, ax=ax,
)
ax.set_axis_off()

The ID column can be used to allocate nodes to the nearest school,
similarly as is done in the map below.

In [ ]:
ax = art_school.plot(markersize=1, color='black', zorder=2)
network.nodes_df.plot(
    nearest_art_school["poi1"], markersize=.01, categorical=True, cmap="Spectral", ax=ax,
)
ax.set_axis_off()

This is, in principle, a Voronoi tessellation based on the distance
along street network rather than a typical Euclidean distance.

### Isochrones

Since we have started allocated space to locations based on distance,
let’s stick to it for a bit and explore isochrones. An isochrone
delineates an area reachable via the network within a set time or
distance. `pandarm` can help us define isochrones relatively easily
(though we need to do some manual coding).

We’re going to generate isochrones reflecting approximate 15 minute
walking distance (~1200 meters) from each art school. To do so, we first
need to understand which node in the network is the nearest one to each
of our art schools, as the analysis is tied to existing nodes.

In [ ]:
art_schoool_nodes = network.get_node_ids(
    art_school.geometry.x, 
    art_school.geometry.y,
)
art_schoool_nodes.head()

The resulting series gives as an ID of the node used to represent
approximate location of each art school. With this, you can find all
other nodes in the specified range, which reflects an isochrone.

In [ ]:
isochrones = network.nodes_in_range(art_schoool_nodes, radius=1000)
isochrones

The output DataFrame is not the most used friendly, but you can see that
the `source` column contains repeated IDs of source nodes, with all
other nodes within the `radius` and an exact distance to them. Getting
the isochrones as polygons, is then a simple GeoPandas operation.

In [ ]:
isochrones_poly = (
    network.nodes_df.loc[isochrones["destination"]]
    .dissolve(isochrones["source"].values)
    .concave_hull(ratio=0.2)
)
isochrones_poly.head()

You can now explore the outcome.

In [ ]:
m = isochrones_poly.explore()
art_school.explore(m=m, color='red')

If that is of interest, you can furhter analyse and disect the nodes
forming the isochrones as they all have the distance. For example, can
peel outer nodes to form smaller isochrones without recomputing the
routes.

In [ ]:
ax = network.nodes_df.loc[isochrones["destination"]].plot(isochrones['length'].values, markersize=.1, legend=True)
ax.set_axis_off()

### Network-based aggregations

`pandarm` is also well suited for data aggregations along the network.
That can capture accessible population, accessible school capacity or
similar questions. Our school dataset contains information about the
school capacity in the `KAPACITA` column, so we can try to understand
how does the capacity affect the access.

We also have population information linked to census output areas, so we
can even ask the question of school capacity normalised by the local
population.

First, we need to assing variables from onto the network. Since we
cannot add additional nodes to the network, we find the nearest existing
node to each point with a value and assign the values to nodes.

In [ ]:
art_node_ids = network.get_node_ids(art_school.geometry.x, art_school.geometry.y)
pop_node_ids = network.get_node_ids(prague.centroid.x, prague.centroid.y)

network.set(art_node_ids, variable=art_school["KAPACITA"], name="art_school_capacity")
network.set(pop_node_ids, variable=prague["n_people"], name="population")

We are interested in the number of spots in an art school within an
accessible range per 1000 people. By accessible we mean approximately 30
minutes walking as people are more willing to walk further for
specialised needs like this one. However, the furher away we are, the
less people are willing to take the trip. We can therefore start with
allocating accessible population to each node as a distance-weighted
sum.

In [ ]:
accessible_population = network.aggregate(
    distance=2400,
    func="sum",
    decay="linear",
    name="population",
)
accessible_population.head()

This aggregation results in a spatial distribution of accessible
population highlighting areas of high population density but a
relatively smoothed one.

In [ ]:
ax = network.nodes_df.plot(accessible_population, markersize=.01, legend=True)
ax.set_axis_off()

We can now do the same aggregation for the capacity of art schools.
Again, using the linear decay to account for difference in willingness
to walk with the increasing distance.

In [ ]:
accessible_capacity = network.aggregate(
    distance=2400, 
    func="sum", 
    decay="linear", 
    name="art_school_capacity",
)

The acessible capacity then takes into the account locations of art
schools, their individual capacity and ability to reach them via street
network.

In [ ]:
ax = art_school.plot(color='red', zorder=2, markersize=2)
ax = network.nodes_df.plot(accessible_capacity, markersize=.01, legend=True, ax=ax)
ax.set_axis_off()

Now, we can simply normalise the accessible capacity by accessible
population to get an approximation of general art school accessibilty.

In [ ]:
normalised_capacity = (
    accessible_capacity / accessible_population
).replace(np.inf, np.nan) * 1000

The results show hotspots of higher accessibiulity and coldspots of
lower one, with some peripheral locations indicating high accessibility,
but those schools are quite likely serving a wider suburban region.

In [ ]:
ax = art_school.plot(color='red', zorder=2, markersize=2)
network.nodes_df.plot(normalised_capacity, markersize=.01, legend=True, vmax=100, ax=ax)
ax.set_axis_off()

Keep in mind that this example is an oversimplification as quite often
people will travel across the city using public transport, which
significantly changes the perception of distance, as we will explore
below.

### Network interaction graphs

The entire idea of accessibility from one place to the other through the
street network can be translated into the spatial weights encoding
spatial interaction. `libpysal` has a built-in method to derive such
weights from a provided `pandarm` network.

In [ ]:
# this needs new libpysal
travel_cost_graph = graph.Graph.build_travel_cost(
    prague.centroid,
    network=network,
    threshold=2000,
    kernel="bisquare",
)
travel_cost_graph

The resulting `Graph` can be used as any other, for example to derive
spatial lag.

In [ ]:
lagged_population = travel_cost_graph.lag(prague["n_people"])
prague.explore(lagged_population)

## Multi-modal movement

Up until now, we were assuming movement through the street network on
foot. However, urban environments are often served by dense networks of
public transport, some of which runs on streets but some which does not.
To provide more realistic estimation of accessibility and network-based
interaction, we shall combine different modes of transport and make full
use of public transport. To do so, we need to rely on data capturing
public transport schedules in a format named [General Transit Feed
Specification (GTFS)](https://gtfs.org). This contains all the
information about routes, stops, schedules, transfers, agencies running
individual routes and other aspects of public transport systems. For
Prague, you [can get](https://pid.cz/o-systemu/opendata/) all this
information for two weeks ahead.

`pandarm` cannot model multi-modal movement but we can use another
tool - `r5py`. That works similarly to `pandarm` by creating a network
object first. Apart from the GTFS data, you need to provide a local
extract from OpenStreetMap for `r5py` to use its street network
representation to route along.

In [ ]:
transport_network = r5py.TransportNetwork(
    osm_pbf="https://martinfleischmann.net/sds/spatial_networks/data/praha-261020.osm.pbf",
    gtfs=["https://martinfleischmann.net/sds/spatial_networks/data/PID_GTFS.zip"],
    allow_errors=True,
)

### Isochrones

As with `pandarm`, we can delineate isochrones as areas accessible
within a given time from one or more origins. The two packages work a
bit differently, so the APIs do not match. Nevertheless, the concepts
are similar. In `r5py`, getting the isochrones is easier as there is a
dedicated way of doing so via `r5py.Isochrones`.

Let’s try to define isochrones around art schools in Prague as we did
before, this time using public transport information.

In [ ]:
art_school['id'] = art_school.NAZEV_ZAR

isochrones = r5py.Isochrones(
    transport_network,
    origins=art_school,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    isochrones=[5, 10, 15],
)
isochrones

The output is a MultiLineString per each time, proviging boundaries of
each isochrone.

In [ ]:
isochrones['travel_time'] = isochrones['travel_time'].astype(int)
ax = isochrones.plot(
    'travel_time', legend=True, legend_kwds={"label": "Distance [seconds]"},
)
ax.set_axis_off()

As you can see above, the isochrones are joined together - we get areas
accessible from one or more art schools, not a boundary per school. It
is often easier to work with isochrones as Polygons, so let’s build
areas from these LineStrings.

In [ ]:
isochrones["polygons"] = shapely.build_area(
    isochrones.geometry 
).set_crs(isochrones.crs)
isochrones.set_geometry("polygons").explore('travel_time')

### Travel-time matrix

In [ ]:
prague['id'] = prague.index

origins = prague.set_geometry(prague.centroid)
destinations = origins.loc[["Albertov"]]

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
)
travel_times

In [ ]:
ax = prague.plot(travel_times.travel_time.values, legend=True)
ax.set_axis_off()

Full matrix

In [ ]:
destinations = prague.set_geometry(prague.centroid)
origins = destinations

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
    max_time=datetime.timedelta(hours=1),
)
travel_times

In [ ]:
transit_graph = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="travel_time",
)

In [ ]:
transit_graph["Albertov"]

In [ ]:
travel_times["weights"] = kernels.kernel(travel_times["travel_time"], bandwidth=60, kernel="bisquare")
travel_times

In [ ]:
transit_graph_weighted = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="weights",
)

In [ ]:
transit_graph_weighted["Albertov"]

In [ ]:
ax = prague.assign(w=transit_graph_weighted["Albertov"]).plot('w', legend=True)
ax.set_axis_off()

In [ ]:
lagged_population = transit_graph_weighted.lag(prague["n_people"])
prague.explore(lagged_population)